# CSCI 611 Assignment 3: Small Object Detection with YOLO
## Traffic sign / traffic light detection — full pipeline

1. Set up YOLO and test on sample image  
2. Prepare dataset (Mapillary-style JSON → YOLO format), train/val split  
3. Baseline: run pre-trained YOLO on test set  
4. Fine-tune YOLO on traffic sign dataset  
5. Evaluate fine-tuned model and compare with pre-trained  
6. Optional: confidence/NMS tuning

---
## 2. Imports and paths

In [9]:
import json
import shutil
from pathlib import Path
import random

import numpy as np
if np.__version__.startswith("2"):
    raise RuntimeError(
        "NumPy 2.x is loaded. Restart the kernel: Kernel → Restart. "
        "Then run the install cell (section 1) and restart again, then run from here."
    )
import cv2
from ultralytics import YOLO

ROOT = Path(".").resolve()
DATA_ROOT = ROOT / "Data"
YOLO_DS = ROOT / "yolo_dataset"
RESULTS_DIR = ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

print("ROOT:", ROOT)
print("DATA_ROOT:", DATA_ROOT)
print("YOLO_DS:", YOLO_DS)
print("NumPy:", np.__version__, "(OK for training)")

ROOT: C:\Studies\Semester\Spring 2026\CSCI 611\A03
DATA_ROOT: C:\Studies\Semester\Spring 2026\CSCI 611\A03\Data
YOLO_DS: C:\Studies\Semester\Spring 2026\CSCI 611\A03\yolo_dataset
NumPy: 1.26.4 (OK for training)


---
## 3. Collect image–annotation pairs and build class list
Scans `Data` for all `.jpg` + `.json` pairs (same base name); builds a single class list from all labels.

In [10]:
def find_image_annotation_pairs(root: Path):
    """Find all (image_path, json_path) where base names match. Prefer .jpg."""
    pairs = []
    root = Path(root)
    for jpath in root.rglob("*.json"):
        if "fully_annotated" in jpath.name:
            continue
        base = jpath.parent / jpath.stem
        for ext in [".jpg", ".png"]:
            ipath = base.with_suffix(ext)
            if ipath.exists():
                pairs.append((ipath, jpath))
                break
    return pairs

pairs = find_image_annotation_pairs(DATA_ROOT)
print(f"Found {len(pairs)} image–annotation pairs.")

# Build sorted class list from all JSONs
all_labels = set()
for ipath, jpath in pairs:
    with open(jpath) as f:
        data = json.load(f)
    for obj in data.get("objects", []):
        lbl = obj.get("label")
        if lbl:
            all_labels.add(lbl)

classes = sorted(all_labels)
class_to_id = {c: i for i, c in enumerate(classes)}
print(f"Number of classes: {len(classes)}")
print("Classes:", classes[:10], "..." if len(classes) > 10 else classes)

Found 18 image–annotation pairs.
Number of classes: 37
Classes: ['complementary--chevron-left--g3', 'complementary--chevron-right--g1', 'complementary--obstacle-delineator--g2', 'information--children--g1', 'information--parking--g1', 'information--pedestrians-crossing--g1', 'other-sign', 'regulatory--go-straight-or-turn-left--g3', 'regulatory--height-limit--g1', 'regulatory--keep-left--g1'] ...


---
## 4. Convert to YOLO format
Each image gets a `.txt` with one line per object: `class_id x_center y_center width height` (normalized 0–1).

In [11]:
def json_to_yolo_line(bbox, img_w, img_h, class_to_id, label):
    xmin = bbox["xmin"]
    ymin = bbox["ymin"]
    xmax = bbox["xmax"]
    ymax = bbox["ymax"]
    x_center = (xmin + xmax) / 2 / img_w
    y_center = (ymin + ymax) / 2 / img_h
    w = (xmax - xmin) / img_w
    h = (ymax - ymin) / img_h
    cid = class_to_id.get(label)
    if cid is None:
        return None
    return f"{cid} {x_center:.6f} {y_center:.6f} {w:.6f} {h:.6f}"

def convert_pair_to_yolo(ipath, jpath, labels_dir: Path, class_to_id):
    img = cv2.imread(str(ipath))
    if img is None:
        return False
    img_h, img_w = img.shape[:2]
    with open(jpath) as f:
        data = json.load(f)
    # Some JSONs have width/height; prefer image shape
    if "width" in data and "height" in data:
        img_w, img_h = data["width"], data["height"]
    lines = []
    for obj in data.get("objects", []):
        bbox = obj.get("bbox")
        label = obj.get("label")
        if not bbox or not label:
            continue
        line = json_to_yolo_line(bbox, img_w, img_h, class_to_id, label)
        if line:
            lines.append(line)
    out_txt = labels_dir / (ipath.stem + ".txt")
    out_txt.parent.mkdir(parents=True, exist_ok=True)
    with open(out_txt, "w") as f:
        f.write("\n".join(lines))
    return True

# Train/val split and YOLO dataset creation
TRAIN_RATIO = 0.85
random.seed(42)
shuffled = pairs.copy()
random.shuffle(shuffled)
n_train = max(1, int(len(shuffled) * TRAIN_RATIO))
train_pairs = shuffled[:n_train]
val_pairs = shuffled[n_train:]
print(f"Train: {len(train_pairs)}, Val (test): {len(val_pairs)}")

for split_name, pair_list in [("train", train_pairs), ("val", val_pairs)]:
    im_dir = YOLO_DS / "images" / split_name
    lb_dir = YOLO_DS / "labels" / split_name
    im_dir.mkdir(parents=True, exist_ok=True)
    lb_dir.mkdir(parents=True, exist_ok=True)
    for ipath, jpath in pair_list:
        shutil.copy2(ipath, im_dir / ipath.name)
        convert_pair_to_yolo(ipath, jpath, lb_dir, class_to_id)

print("YOLO dataset structure created at", YOLO_DS)

Train: 15, Val (test): 3
YOLO dataset structure created at C:\Studies\Semester\Spring 2026\CSCI 611\A03\yolo_dataset


---
## 5. Create data.yaml

In [12]:
yaml_path = YOLO_DS / "data.yaml"
yaml_content = {
    "path": str(YOLO_DS.resolve()),
    "train": "images/train",
    "val": "images/val",
    "nc": len(classes),
    "names": classes,
}
with open(yaml_path, "w") as f:
    f.write("path: " + str(YOLO_DS.resolve()) + "\n")
    f.write("train: images/train\n")
    f.write("val: images/val\n")
    f.write("nc: " + str(len(classes)) + "\n")
    f.write("names: " + json.dumps(classes) + "\n")

print("data.yaml written to", yaml_path)
print(open(yaml_path).read()[:500])

data.yaml written to C:\Studies\Semester\Spring 2026\CSCI 611\A03\yolo_dataset\data.yaml
path: C:\Studies\Semester\Spring 2026\CSCI 611\A03\yolo_dataset
train: images/train
val: images/val
nc: 37
names: ["complementary--chevron-left--g3", "complementary--chevron-right--g1", "complementary--obstacle-delineator--g2", "information--children--g1", "information--parking--g1", "information--pedestrians-crossing--g1", "other-sign", "regulatory--go-straight-or-turn-left--g3", "regulatory--height-limit--g1", "regulatory--keep-left--g1", "regulatory--keep-right--g1", "regulatory--keep-right--


---
## 6. Save annotated dataset samples (for deliverables)
Draw ground-truth boxes on a few train images and save to `results/annotated_samples`.

In [13]:
annotated_out = RESULTS_DIR / "annotated_samples"
annotated_out.mkdir(exist_ok=True)
train_dir = YOLO_DS / "images" / "train"
if not train_dir.exists():
    print("yolo_dataset/images/train does not exist yet. Run the conversion/split cell (section 4) first.")
else:
    train_imgs = [p for p in train_dir.iterdir() if p.suffix.lower() in (".jpg", ".png")][:3]
    if not train_imgs:
        print("No train images found to annotate.")
    else:
        for imp in train_imgs:
            img = cv2.imread(str(imp))
            if img is None:
                continue
            h, w = img.shape[:2]
            lbl_path = YOLO_DS / "labels" / "train" / (imp.stem + ".txt")
            if not lbl_path.exists():
                continue
            with open(lbl_path) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) < 5:
                        continue
                    cid, xc, yc, bw, bh = float(parts[0]), float(parts[1]), float(parts[2]), float(parts[3]), float(parts[4])
                    x1 = int((xc - bw / 2) * w)
                    y1 = int((yc - bh / 2) * h)
                    x2 = int((xc + bw / 2) * w)
                    y2 = int((yc + bh / 2) * h)
                    cv2.rectangle(img, (x1, y1), (x2, y2), (0, 255, 0), 2)
                    name = classes[int(cid)] if int(cid) < len(classes) else str(int(cid))
                    cv2.putText(img, name[:20], (x1, y1 - 5), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 0), 1)
            out_path = annotated_out / imp.name
            cv2.imwrite(str(out_path), img)
            print("Saved", out_path)
        print("Annotated samples in", annotated_out)

Saved C:\Studies\Semester\Spring 2026\CSCI 611\A03\results\annotated_samples\1IHgtTsrvIPhEDnYfdxKyQ.jpg
Saved C:\Studies\Semester\Spring 2026\CSCI 611\A03\results\annotated_samples\5nYy-qqD3nRlwtomubxRrA.jpg
Saved C:\Studies\Semester\Spring 2026\CSCI 611\A03\results\annotated_samples\B1GzCp0arIYNkaS85EjoIg.jpg
Annotated samples in C:\Studies\Semester\Spring 2026\CSCI 611\A03\results\annotated_samples


---
## 7. Test pre-trained YOLO on one sample image

In [14]:
sample_img = list((YOLO_DS / "images" / "train").glob("*.jpg")) or list((YOLO_DS / "images" / "train").glob("*.png"))
sample_img = sample_img[0] if sample_img else None
if sample_img:
    model_pretrained = YOLO("yolov8n.pt")
    results = model_pretrained(str(sample_img))
    results[0].show()
    results[0].save(str(RESULTS_DIR / "sample_pretrained.jpg"))
    print("Saved to results/sample_pretrained.jpg")
else:
    print("No sample image found.")


image 1/1 C:\Studies\Semester\Spring 2026\CSCI 611\A03\yolo_dataset\images\train\1IHgtTsrvIPhEDnYfdxKyQ.jpg: 480x640 1 bus, 1 truck, 188.3ms
Speed: 7.7ms preprocess, 188.3ms inference, 3.8ms postprocess per image at shape (1, 3, 480, 640)
Saved to results/sample_pretrained.jpg


---
## 8. Baseline: run pre-trained YOLO on test (val) set
Pre-trained YOLO detects COCO classes (e.g. traffic light). We run on val images and save results for comparison.

In [15]:
val_dir = YOLO_DS / "images" / "val"
model_pretrained = YOLO("yolov8n.pt")
baseline_results = model_pretrained.predict(
    source=str(val_dir),
    save=True,
    project=str(RESULTS_DIR),
    name="pretrained_val",
    exist_ok=True,
    conf=0.5,
)
print(f"Pre-trained inference on val set. Outputs in {RESULTS_DIR / 'pretrained_val'}")
for r in baseline_results:
    if r and len(r) > 0 and r[0].boxes is not None:
        n = len(r[0].boxes)
        print(f"  {Path(r[0].path).name}: {n} detections")


image 1/3 C:\Studies\Semester\Spring 2026\CSCI 611\A03\yolo_dataset\images\val\1b8VQaHs_3W1qnlenEQudw.jpg: 384x640 (no detections), 162.6ms
image 2/3 C:\Studies\Semester\Spring 2026\CSCI 611\A03\yolo_dataset\images\val\_yavKJikt1ztFz48JNbRdg.jpg: 480x640 3 cars, 1 traffic light, 149.2ms
image 3/3 C:\Studies\Semester\Spring 2026\CSCI 611\A03\yolo_dataset\images\val\jm7I1Z22iwg3NVVuAxnFzg.jpg: 480x640 1 person, 3 cars, 1 truck, 286.7ms
Speed: 48.2ms preprocess, 199.5ms inference, 1.8ms postprocess per image at shape (1, 3, 480, 640)
Results saved to C:\Studies\Semester\Spring 2026\CSCI 611\A03\results\pretrained_val
Pre-trained inference on val set. Outputs in C:\Studies\Semester\Spring 2026\CSCI 611\A03\results\pretrained_val
  _yavKJikt1ztFz48JNbRdg.jpg: 1 detections
  jm7I1Z22iwg3NVVuAxnFzg.jpg: 1 detections


---
## 9. Train (fine-tune) YOLO on traffic sign dataset
Uses higher resolution (640; optional 1280) and augmentation as per assignment.

In [16]:
model = YOLO("yolov8n.pt")
train_results = model.train(
    data=str(yaml_path),
    epochs=30,
    imgsz=640,
    batch=16,
    project=str(RESULTS_DIR),
    name="train",
    exist_ok=True,
    patience=10,
)
best_weights_path = RESULTS_DIR / "train" / "weights" / "best.pt"
print(f"Training complete. Best weights saved to: {best_weights_path}")

New https://pypi.org/project/ultralytics/8.4.22 available  Update with 'pip install -U ultralytics'
Ultralytics 8.4.21  Python-3.12.3 torch-2.10.0+cpu CPU (11th Gen Intel Core i7-1185G7 @ 3.00GHz)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=C:\Studies\Semester\Spring 2026\CSCI 611\A03\yolo_dataset\data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=30, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937

---
## 10. Run fine-tuned model on test (val) set

In [17]:
best_weights = RESULTS_DIR / "train" / "weights" / "best.pt"
if not best_weights.exists():
    best_weights = ROOT / "runs" / "detect" / "train" / "weights" / "best.pt"
if not best_weights.exists():
    print("Best weights not found. Check RESULTS_DIR/train/ or runs/detect/train/")
else:
    model_finetuned = YOLO(str(best_weights))
    finetuned_out = model_finetuned.predict(
        source=str(YOLO_DS / "images" / "val"),
        save=True,
        project=str(RESULTS_DIR),
        name="finetuned_val",
        exist_ok=True,
        conf=0.5,
        iou=0.45,
    )
    print(f"Fine-tuned inference saved to {RESULTS_DIR / 'finetuned_val'}")


image 1/3 C:\Studies\Semester\Spring 2026\CSCI 611\A03\yolo_dataset\images\val\1b8VQaHs_3W1qnlenEQudw.jpg: 384x640 (no detections), 80.5ms
image 2/3 C:\Studies\Semester\Spring 2026\CSCI 611\A03\yolo_dataset\images\val\_yavKJikt1ztFz48JNbRdg.jpg: 480x640 (no detections), 91.2ms
image 3/3 C:\Studies\Semester\Spring 2026\CSCI 611\A03\yolo_dataset\images\val\jm7I1Z22iwg3NVVuAxnFzg.jpg: 480x640 (no detections), 85.7ms
Speed: 4.5ms preprocess, 85.8ms inference, 1.5ms postprocess per image at shape (1, 3, 480, 640)
Results saved to C:\Studies\Semester\Spring 2026\CSCI 611\A03\results\finetuned_val
Fine-tuned inference saved to C:\Studies\Semester\Spring 2026\CSCI 611\A03\results\finetuned_val


---
## 11. Compare pre-trained vs fine-tuned
Summarize detection counts and optionally tune confidence threshold.

In [18]:
def run_and_count(model, imgs, conf=0.5):
    counts = []
    for imp in imgs:
        r = model(str(imp), conf=conf, verbose=False)
        n = len(r[0].boxes) if r and len(r) > 0 and r[0].boxes is not None else 0
        counts.append((imp.name, n))
    return counts

val_images = list((YOLO_DS / "images" / "val").glob("*.*"))
val_images = [p for p in val_images if p.suffix.lower() in (".jpg", ".png")]

pretrained = run_and_count(YOLO("yolov8n.pt"), val_images)
print("Pre-trained (COCO) detections (conf>=0.5):")
for name, n in pretrained:
    print(f"  {name}: {n}")

best_weights = RESULTS_DIR / "train" / "weights" / "best.pt"
if not best_weights.exists():
    best_weights = ROOT / "runs" / "detect" / "train" / "weights" / "best.pt"
if best_weights.exists():
    finetuned = run_and_count(YOLO(str(best_weights)), val_images)
    print("\nFine-tuned (traffic signs) detections (conf>=0.5):")
    for name, n in finetuned:
        print(f"  {name}: {n}")
    print("\n(Pre-trained detects COCO classes e.g. traffic light; fine-tuned detects your traffic sign classes.)")
else:
    print("\nFine-tuned model not available yet (training not completed or failed).")

Pre-trained (COCO) detections (conf>=0.5):
  1b8VQaHs_3W1qnlenEQudw.jpg: 0
  jm7I1Z22iwg3NVVuAxnFzg.jpg: 5
  _yavKJikt1ztFz48JNbRdg.jpg: 4

Fine-tuned (traffic signs) detections (conf>=0.5):
  1b8VQaHs_3W1qnlenEQudw.jpg: 0
  jm7I1Z22iwg3NVVuAxnFzg.jpg: 0
  _yavKJikt1ztFz48JNbRdg.jpg: 0

(Pre-trained detects COCO classes e.g. traffic light; fine-tuned detects your traffic sign classes.)


---
## 12. Optional: confidence threshold and NMS
Re-run fine-tuned model with different conf/iou to reduce false positives or catch more small objects.

In [19]:
best_weights = RESULTS_DIR / "train" / "weights" / "best.pt"
if not best_weights.exists():
    best_weights = ROOT / "runs" / "detect" / "train" / "weights" / "best.pt"
if best_weights.exists():
    model_ft = YOLO(str(best_weights))
    # Lower confidence to reduce false negatives on small objects
    model_ft.predict(
        source=str(YOLO_DS / "images" / "val"),
        save=True,
        project=str(RESULTS_DIR),
        name="finetuned_conf0.3",
        exist_ok=True,
        conf=0.3,
        iou=0.45,
    )
    print("Saved results with conf=0.3 to results/finetuned_conf0.3")
else:
    print("Fine-tuned model not available yet (training not completed or failed).")


image 1/3 C:\Studies\Semester\Spring 2026\CSCI 611\A03\yolo_dataset\images\val\1b8VQaHs_3W1qnlenEQudw.jpg: 384x640 (no detections), 73.3ms
image 2/3 C:\Studies\Semester\Spring 2026\CSCI 611\A03\yolo_dataset\images\val\_yavKJikt1ztFz48JNbRdg.jpg: 480x640 (no detections), 82.6ms
image 3/3 C:\Studies\Semester\Spring 2026\CSCI 611\A03\yolo_dataset\images\val\jm7I1Z22iwg3NVVuAxnFzg.jpg: 480x640 (no detections), 77.8ms
Speed: 2.9ms preprocess, 77.9ms inference, 1.7ms postprocess per image at shape (1, 3, 480, 640)
Results saved to C:\Studies\Semester\Spring 2026\CSCI 611\A03\results\finetuned_conf0.3
Saved results with conf=0.3 to results/finetuned_conf0.3
